# Week 3: Advanced SQL Analysis using Subqueries, CTEs & Window Functions
**Celebal Excellence Internship (CEI) 2026**

Objective: Analyze the Superstore dataset using advanced SQL concepts.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import sqlite3

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df.head()

## Step 3: Create SQLite Database

In [ ]:
conn = sqlite3.connect("superstore.db")
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)
print("Table created successfully.")

## Step 4: Create Required Tables

### Customers Table

In [ ]:
query = '''
CREATE TABLE customers AS
SELECT DISTINCT
[Customer ID],[Customer Name],Segment,Country,City,State,Region
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS customers")
conn.execute(query)
pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)

### Orders Table

In [ ]:
query = '''
CREATE TABLE orders AS
SELECT DISTINCT
[Order ID],[Order Date],[Ship Date],[Ship Mode],[Customer ID]
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS orders")
conn.execute(query)
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

### Products Table

In [ ]:
query = '''
CREATE TABLE products AS
SELECT DISTINCT
[Product ID],Category,[Sub-Category],[Product Name]
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS products")
conn.execute(query)
pd.read_sql("SELECT * FROM products LIMIT 5;", conn)

## Orders Above Average Sales (Subquery)

In [ ]:
query = """SELECT * FROM superstore_raw
WHERE Sales > (SELECT AVG(Sales) FROM superstore_raw);"""
pd.read_sql(query, conn)

## Highest Sales Order Per Customer (Subquery)

In [ ]:
query = """SELECT * FROM superstore_raw s
WHERE Sales=(SELECT MAX(Sales) FROM superstore_raw
WHERE [Customer ID]=s.[Customer ID]);"""
pd.read_sql(query, conn)

## Total Sales Per Customer (CTE)

In [ ]:
query = """WITH customer_sales AS (
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT * FROM customer_sales;"""
pd.read_sql(query, conn)

## Customers Above Average Sales (CTE + Subquery)

In [ ]:
query = """WITH customer_sales AS (
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT * FROM customer_sales
WHERE Total_Sales>(SELECT AVG(Total_Sales) FROM customer_sales);"""
pd.read_sql(query, conn)

## Rank Customers (Window Function)

In [ ]:
query = """SELECT [Customer Name],
SUM(Sales) AS Total_Sales,
RANK() OVER(ORDER BY SUM(Sales) DESC) AS Sales_Rank
FROM superstore_raw
GROUP BY [Customer Name];"""
pd.read_sql(query, conn)

## Row Number Within Each Customer

In [ ]:
query = """SELECT [Customer Name],[Order ID],Sales,
ROW_NUMBER() OVER(
PARTITION BY [Customer Name]
ORDER BY Sales DESC) AS Order_Number
FROM superstore_raw;"""
pd.read_sql(query, conn)

## Top 3 Customers

In [ ]:
query = """SELECT * FROM(
SELECT [Customer Name],
SUM(Sales) Total_Sales,
RANK() OVER(ORDER BY SUM(Sales) DESC) Sales_Rank
FROM superstore_raw
GROUP BY [Customer Name])
WHERE Sales_Rank<=3;"""
pd.read_sql(query, conn)

## Final Combined Query

In [ ]:
query = """WITH customer_sales AS(
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT [Customer Name],Total_Sales,
RANK() OVER(ORDER BY Total_Sales DESC) AS Customer_Rank
FROM customer_sales;"""
pd.read_sql(query, conn)

## Top 5 Customers

In [ ]:
query = """SELECT [Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer Name]
ORDER BY Total_Sales DESC
LIMIT 5;"""
pd.read_sql(query, conn)

## Bottom 5 Customers

In [ ]:
query = """SELECT [Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer Name]
ORDER BY Total_Sales ASC
LIMIT 5;"""
pd.read_sql(query, conn)

## Customers With One Order

In [ ]:
query = """SELECT [Customer Name],COUNT(DISTINCT [Order ID]) Orders
FROM superstore_raw
GROUP BY [Customer Name]
HAVING Orders=1;"""
pd.read_sql(query, conn)

## Highest Order Value Per Customer

In [ ]:
query = """SELECT [Customer Name],MAX(Sales) Highest_Order
FROM superstore_raw
GROUP BY [Customer Name];"""
pd.read_sql(query, conn)

# Conclusion

## Key Insights
- Used Subqueries for comparative analysis.
- Applied CTEs for reusable aggregations.
- Used Window Functions (RANK and ROW_NUMBER) for ranking and partitioning.
- Combined CTEs and Window Functions to produce customer sales insights.
- Answered business-oriented questions from the Superstore dataset.
